# Pre-sprint frozen-history audit
Prepared before the Sprint. Select an A100 runtime. Read the CU/hour rate first; allow up to 3 hours (budget 30 CU at 10 CU/hour). Stop if the unchanged-prompt reconstruction fails broadly. Upload the supplied audit-source.zip; no Git push is needed. Model access requires your own authorized Hugging Face account. Keep this browser open and download results before disconnecting.

In [ ]:
from google.colab import files
import zipfile, os
uploaded = files.upload()
assert "audit-source.zip" in uploaded
with zipfile.ZipFile("audit-source.zip") as z:
    z.extractall("/content/frozen-audit")
os.chdir("/content/frozen-audit")


In [ ]:
!python scripts/verify_provenance.py


In [ ]:
%pip -q install transformers accelerate numpy


In [ ]:
from huggingface_hub import login
login()  # user completes login; do not put a token in saved cells


In [ ]:
import torch, json, time
assert torch.cuda.is_available(), "Select a GPU runtime"
print(torch.cuda.get_device_name(0))
print("VRAM GB:", torch.cuda.get_device_properties(0).total_memory / 2**30)
m=json.load(open("provenance/checkpoints.json"))
assert len(m["checkpoints"]) == 30
print("Checkpoints:", len(m["checkpoints"]))


In [ ]:
# Resolve once; save this SHA to reproduce this run. Historical weights remain unverified.
from huggingface_hub import HfApi
MODEL = "meta-llama/Llama-3.1-8B-Instruct"
REVISION = HfApi().model_info(MODEL).sha
print("Pinned revision:", REVISION)
# Shell magic streams per-checkpoint PASS/failure lines into this cell; subprocess.run may not.
!python scripts/frozen_audit.py run --manifest provenance/checkpoints.json --model {MODEL} --revision {REVISION} --out results/presprint --device cuda


In [ ]:
!python scripts/frozen_audit.py report --results results/presprint --out results/paired.csv


In [ ]:
import shutil
shutil.make_archive("/content/frozen-audit-results", "zip", "results")
files.download("/content/frozen-audit-results.zip")


After download, disconnect and delete the runtime to stop GPU consumption. Return the results ZIP for manual text review and the paired diagnostic analysis. Failed reconstructions remain in the output; do not label them successful audits.